In [0]:
# ── CONFIG ───────────────────────────────────────────────────────────────────
 
CATALOG    = "clutchlytics"
SOURCE     = f"{CATALOG}.silver.fctGames"
GOLD_TABLE = f"{CATALOG}.gold.nhl_gold_home_ice"
 
LEAGUE              = "nhl"
SPORT               = "hockey"
HISTORICAL_NORM     = 0.59
BELOW_THRESHOLD     = 0.55
 
print(f"Source             : {SOURCE}")
print(f"Target             : {GOLD_TABLE}")
print(f"Historical norm    : {HISTORICAL_NORM}")
print(f"Below threshold    : {BELOW_THRESHOLD}")

In [0]:
# ── READ SOURCE ───────────────────────────────────────────────────────────────
 
from pyspark.sql import functions as F
from datetime import datetime, timezone
 
fct_df = (
    spark.table(SOURCE)
    .filter(
        (F.col("league") == LEAGUE) &
        (F.col("clutch_game_id").isNotNull())  # exclude incomplete R2 rows
    )
)

# ── Load dimGames to get season_type and game_date ──
dim_games = (
    spark.table(f"{CATALOG}.silver.dimGames")
    .filter(F.col("league") == LEAGUE)
    .select(
        F.col("clutch_game_id").alias("dim_game_id"),
        F.col("season_type"),
        F.col("game_date"),
    )
)

# ── Join to get season_type and game_date ──
fct_df = (
    fct_df
    .join(
        dim_games,
        fct_df.clutch_game_id == dim_games.dim_game_id,
        how="left"
    )
    .drop("dim_game_id")
)
 
print(f"fctGames rows ({LEAGUE}, complete): {fct_df.count()}")

In [0]:
# ── AGGREGATE BY ROUND ────────────────────────────────────────────────────────
 
ingested_at = datetime.now(timezone.utc).isoformat()
 
gold_df = (
    fct_df
    .groupBy("round", "season_type")
    .agg(
        F.count("*").alias("games_played"),
 
        # ── Home/away wins ──
        F.sum(F.when(F.col("home_winner") == True, 1).otherwise(0))
         .alias("home_wins"),
        F.sum(F.when(F.col("away_winner") == True, 1).otherwise(0))
         .alias("away_wins"),
 
        # ── OT breakdown ──
        F.sum(F.when(F.col("went_to_ot") == True, 1).otherwise(0))
         .alias("ot_games"),
        F.sum(
            F.when(
                (F.col("went_to_ot") == True) & (F.col("home_winner") == True), 1
            ).otherwise(0)
        ).alias("ot_home_wins"),
 
        # ── Scoring context ──
        F.round(F.avg(F.col("home_score") + F.col("away_score")), 2)
         .alias("avg_total_goals"),
        F.round(F.avg(F.col("home_score")), 2).alias("avg_home_goals"),
        F.round(F.avg(F.col("away_score")), 2).alias("avg_away_goals"),
 
        # ── Series clinching games ──
        F.sum(F.when(F.col("series_clinched") == True, 1).otherwise(0))
         .alias("clinching_games"),
        F.sum(
            F.when(
                (F.col("series_clinched") == True) & (F.col("home_winner") == True), 1
            ).otherwise(0)
        ).alias("clinching_games_home_won"),
 
        # ── Date range ──
        F.min("game_date").alias("round_start_date"),
        F.max("game_date").alias("round_end_date"),
    )
    # ── Derived metrics ──
    .withColumn(
        "home_win_pct",
        F.round(F.col("home_wins") / F.col("games_played"), 3)
    )
    .withColumn(
        "ot_home_win_pct",
        F.when(
            F.col("ot_games") > 0,
            F.round(F.col("ot_home_wins") / F.col("ot_games"), 3)
        ).otherwise(None)
    )
    .withColumn(
        "clinching_home_win_pct",
        F.when(
            F.col("clinching_games") > 0,
            F.round(F.col("clinching_games_home_won") / F.col("clinching_games"), 3)
        ).otherwise(None)
    )
    .withColumn("historical_nhl_norm", F.lit(HISTORICAL_NORM))
    .withColumn(
        "delta_from_historical",
        F.round(F.col("home_win_pct") - HISTORICAL_NORM, 3)
    )
    .withColumn(
        "below_historical",
        F.col("home_win_pct") < BELOW_THRESHOLD
    )
    # ── Context ──
    .withColumn("sport",       F.lit(SPORT))
    .withColumn("league",      F.lit(LEAGUE))
    .withColumn("ingested_at", F.lit(ingested_at))
    .withColumn("source_table", F.lit("silver.fctGames"))
    # ── Final column order ──
    .select(
        "round",
        "season_type",
        "sport",
        "league",
        "games_played",
        "home_wins",
        "away_wins",
        "home_win_pct",
        "historical_nhl_norm",
        "delta_from_historical",
        "below_historical",
        "ot_games",
        "ot_home_wins",
        "ot_home_win_pct",
        "avg_total_goals",
        "avg_home_goals",
        "avg_away_goals",
        "clinching_games",
        "clinching_games_home_won",
        "clinching_home_win_pct",
        "round_start_date",
        "round_end_date",
        "ingested_at",
        "source_table",
    )
    .orderBy("round")
)
 
print(f"Rows to write: {gold_df.count()}")
gold_df.show(truncate=False)

In [0]:
# ── WRITE TO GOLD ─────────────────────────────────────────────────────────────
# Full overwrite — always reproducible from Silver.
 
(
    gold_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_TABLE)
)
 
print(f"Written to {GOLD_TABLE}")

In [0]:
# ── VALIDATE ─────────────────────────────────────────────────────────────────
 
spark.sql(f"""
    SELECT
        round,
        games_played,
        home_wins,
        away_wins,
        home_win_pct,
        historical_nhl_norm,
        delta_from_historical,
        below_historical,
        ot_games,
        ot_home_win_pct,
        clinching_home_win_pct,
        avg_total_goals
    FROM {GOLD_TABLE}
    ORDER BY round
""").show(truncate=False)

In [0]:
# ── SANITY CHECKS ─────────────────────────────────────────────────────────────
 
checks = spark.sql(f"""
    SELECT
        COUNT(*)                                                AS total_rows,
        SUM(games_played)                                       AS total_games,
        SUM(home_wins + away_wins)                             AS wins_sum,
        COUNT(CASE WHEN home_wins + away_wins
                      != games_played THEN 1 END)              AS win_count_mismatch,
        COUNT(CASE WHEN home_win_pct > 1.0
                    OR home_win_pct < 0.0 THEN 1 END)          AS bad_win_pct,
        COUNT(CASE WHEN ot_home_win_pct IS NULL
                    AND ot_games > 0 THEN 1 END)               AS null_ot_pct_with_games
    FROM {GOLD_TABLE}
""")
 
print("Sanity checks:")
checks.show(truncate=False)